# 🏠 XGBoost Model Training — Real Estate Price Prediction

This notebook trains an XGBoost regression model to predict **property prices** using the synthetic real estate dataset.

**Pipeline:**
1. Load & explore the data
2. Preprocessing & train/test split
3. XGBoost model training with hyperparameter tuning
4. Evaluation — R² and MSE
5. Feature importance visualization

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb

# Plotting style
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

print(f'XGBoost version : {xgb.__version__}')
print(f'Pandas version  : {pd.__version__}')

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('new_synthetic_dataset.csv')

print(f'Dataset shape : {df.shape}')
print(f'\nColumn dtypes:\n{df.dtypes}')
df.head()

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(4)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found ✓')

### Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Price'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (normalised)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['Price']), bins=60, color='teal', edgecolor='white', linewidth=0.4)
axes[1].set_title('log(Price + 1) Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('price_distribution.png', bbox_inches='tight')
plt.show()

### Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 3. Preprocessing & Train/Test Split

In [ ]:
TARGET = 'Price'

# Drop 'Source' — all zeros (no variance)
FEATURES = [c for c in df.columns if c not in [TARGET, 'Source']]

X = df[FEATURES]
y = df[TARGET]

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Target   : {TARGET}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size : {X_train.shape[0]:,}')
print(f'Test  size : {X_test.shape[0]:,}')

## 4. XGBoost Model Training

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)

params = {
    'objective'        : 'reg:squarederror',
    'eval_metric'      : ['rmse', 'mae'],
    'max_depth'        : 6,
    'learning_rate'    : 0.05,
    'n_estimators'     : 500,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_weight' : 3,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'seed'             : 42,
    'tree_method'      : 'hist',      # fast histogram algorithm
    'verbosity'        : 0
}

evals_result = {}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=30,
    evals_result=evals_result,
    verbose_eval=50
)

print(f'\nBest iteration : {model.best_iteration}')

### Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

train_rmse = evals_result['train']['rmse']
test_rmse  = evals_result['test']['rmse']
rounds     = range(1, len(train_rmse) + 1)

ax.plot(rounds, train_rmse, label='Train RMSE', color='steelblue',  linewidth=1.5)
ax.plot(rounds, test_rmse,  label='Test RMSE',  color='tomato',     linewidth=1.5)
ax.axvline(model.best_iteration, linestyle='--', color='gray', alpha=0.7,
           label=f'Best iteration ({model.best_iteration})')

ax.set_title('XGBoost Training Curve (RMSE)', fontsize=13, fontweight='bold')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE')
ax.legend()
plt.tight_layout()
plt.savefig('training_curve.png', bbox_inches='tight')
plt.show()

## 5. Model Evaluation — R² and MSE

In [ ]:
y_pred_train = model.predict(dtrain)
y_pred_test  = model.predict(dtest)

# ── Metrics ──────────────────────────────────────────────
train_mse  = mean_squared_error(y_train, y_pred_train)
test_mse   = mean_squared_error(y_test,  y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse  = np.sqrt(test_mse)
train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)

metrics = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R²'],
    'Train' : [train_mse, train_rmse, train_r2],
    'Test'  : [test_mse,  test_rmse,  test_r2]
})
metrics = metrics.set_index('Metric').round(6)

print('=' * 38)
print('         EVALUATION METRICS')
print('=' * 38)
print(metrics.to_string())
print('=' * 38)

### Predicted vs Actual

In [ ]:
sample_idx = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
y_test_s   = np.array(y_test)[sample_idx]
y_pred_s   = y_pred_test[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(y_test_s, y_pred_s, alpha=0.25, s=8, color='steelblue', rasterized=True)
lo, hi = min(y_test_s.min(), y_pred_s.min()), max(y_test_s.max(), y_pred_s.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_title(f'Predicted vs Actual  (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].legend()

# Residuals
residuals = y_test_s - y_pred_s
axes[1].scatter(y_pred_s, residuals, alpha=0.25, s=8, color='teal', rasterized=True)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', bbox_inches='tight')
plt.show()

## 6. Feature Importance

In [ ]:
importance_types = ['weight', 'gain', 'cover']
importance_labels = {
    'weight': 'Frequency (# splits)',
    'gain'  : 'Gain (avg. info gain)',
    'cover' : 'Cover (avg. samples)'
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'teal', 'darkorange']

for ax, itype, color in zip(axes, importance_types, colors):
    scores = model.get_score(importance_type=itype)
    fi_df  = pd.Series(scores).sort_values(ascending=True)

    bars = ax.barh(fi_df.index, fi_df.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(importance_labels[itype], fontsize=11, fontweight='bold')
    ax.set_xlabel('Score')
    ax.tick_params(axis='y', labelsize=9)

    # Value labels on bars
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.01, bar.get_y() + bar.get_height()/2,
                f'{w:.2f}', va='center', ha='left', fontsize=7.5)

plt.suptitle('XGBoost Feature Importance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

### Top Features by Gain (ranked)

In [ ]:
gain_scores = pd.Series(model.get_score(importance_type='gain')).sort_values(ascending=False)
gain_pct    = (gain_scores / gain_scores.sum() * 100).round(2)

summary = pd.DataFrame({'Gain Score': gain_scores, 'Importance (%)': gain_pct})
print(summary.to_string())

## 7. Save the Model

In [ ]:
model.save_model('xgboost_real_estate.ubj')
print('Model saved → xgboost_real_estate.ubj')

print(f'\n── Final Summary ──────────────────')
print(f'  Train R²  : {train_r2:.5f}')
print(f'  Test  R²  : {test_r2:.5f}')
print(f'  Train MSE : {train_mse:.6f}')
print(f'  Test  MSE : {test_mse:.6f}')
print(f'  Test  RMSE: {test_rmse:.6f}')